In [5]:
# Install the modern stack compatible with Python 3.13
%pip install pandas>=2.2.3 numpy>=2.1.0 seaborn matplotlib scikit-learn tqdm

# Install the deep learning tools we discussed
%pip install transformers>=4.44.0 datasets>=2.20.0 accelerate>=0.34.0
%pip install torch>=2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import torch
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoConfig,AutoModelForCausalLM,AutoModelForSequenceClassification,BertConfig,BertForMaskedLM,TrainingArguments, Trainer, TrainingArguments
from transformers import AutoTokenizer,BertTokenizerFast,DataCollatorForLanguageModeling
from transformers import pipeline
from datasets import load_dataset

from tqdm.auto import tqdm
import math
import time
import os


# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

D:\GitHub\IBM-AI-Engineering-Professional-Certificate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     --------------------------------------- 2.6/455.4 MB 14.6 MB/s eta 0:00:31
      -------------------------------------- 6.3/455.4 MB 16.2 MB/s eta 0:00:28
      ------------------------------------- 10.2/455.4 MB 16.7 MB/s eta 0:00:27
     - ------------------------------------ 13.6/455.4 MB 16.7 MB/s eta 0:00:27
     - ------------------------------------ 17.3/455.4 MB 16.8 MB/s eta 0:00:27
     - ------------------------------------ 20.4/455.4 MB 16.4 MB/s eta 0:00:27
     -- ----------------------------------- 24.1/455.4 MB 16.6 MB/s eta 0:00:27
     -- ----------------------------------- 27.8/455.4 MB 16.6 MB/s eta 0:00:26
     -- ----------------------------------- 31.2/455.4 MB 16.5 MB/s eta 0:00:26
     -- ----------------------------------- 34.3/455.4 MB 16.4 MB/s eta 0:00:26
     --- ---------------------------------- 38.3/455.4 MB 16.6 MB/s eta 0:00:26
     --- ---------------------------------- 41.

  DEPRECATION: Building 'pyspark' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyspark'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\GitHub\\IBM-AI-Engineering-Professional-Certificate\\.venv\\Lib\\site-packages\\tokenizers\\tokenizers.pyd'
Check the permissions.


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/3

In [7]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [8]:
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 680.48it/s, Materializing param=model.decoder.project_out.weight]                   


In [9]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [10]:
result = pipe("this movie was really")

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
result

[{'generated_text': 'this movie was really good but i think it was a little rushed and didnt really have much depth to it. it wasnt that great\nI actually really liked it. It had a lot of potential and it was a really good movie. I think that it was a bit rushed for some people though.'}]

In [13]:
result[0]

{'generated_text': 'this movie was really good but i think it was a little rushed and didnt really have much depth to it. it wasnt that great\nI actually really liked it. It had a lot of potential and it was a really good movie. I think that it was a bit rushed for some people though.'}

In [14]:
result[0]['generated_text']

'this movie was really good but i think it was a little rushed and didnt really have much depth to it. it wasnt that great\nI actually really liked it. It had a lot of potential and it was a really good movie. I think that it was a bit rushed for some people though.'

In [16]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 240127.03 examples/s]


In [17]:
dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [20]:
dataset["train"][400]['text']

" When Mason was injured in warm @-@ ups late in the year , Columbus was without an active goaltender on their roster . To remedy the situation , the team signed former University of Michigan goaltender Shawn Hunwick to a one @-@ day , amateur tryout contract . After being eliminated from the NCAA Tournament just days prior , Hunwick skipped an astronomy class and drove his worn down 2003 Ford Ranger to Columbus to make the game . He served as the back @-@ up to Allen York during the game , and the following day , he signed a contract for the remainder of the year . With Mason returning from injury , Hunwick was third on the team 's depth chart when an injury to York allowed Hunwick to remain as the back @-@ up for the final two games of the year . In the final game of the season , the Blue Jackets were leading the Islanders 7 – 3 with 2 : 33 remaining when , at the behest of his teammates , Head Coach Todd Richards put Hunwick in to finish the game . He did not face a shot . Hunwick w

In [19]:
dataset["train"] = dataset["train"].select([i for i in range(1000)])
dataset["test"] = dataset["test"].select([i for i in range(200)])

In [21]:
output_file_train = "wikitext_dataset_train.txt"
output_file_test = "wikitext_dataset_test.txt"

with open(output_file_train, 'w', encoding='utf-8') as f:
    for example in dataset["train"]:
        f.write(example['text'] + '\n')

with open(output_file_test, 'w', encoding='utf-8') as f:
    for example in dataset['test']:
        f.write(example['text'] + '\n')

In [22]:
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model_name = 'bert-base-uncased'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, is_decoder=True)

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 695.25it/s, Materializing param=cls.predictions.transform.dense.weight]                 
BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
config = BertConfig(
    vocab_size=len(bert_tokenizer.get_vocab()),
    hidden_size=768,
    num_hidden_layers=12,
    num_attention_heads=12,
    intermediate_size=3072,
)

In [24]:
model = BertForMaskedLM(config)

In [25]:
model

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [37]:
# Tokenize dataset dynamically
def tokenize_function(examples):
    return bert_tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Tokenize train and test datasets
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Print tokenized dataset sample
print(tokenized_datasets["train"][0])

# Split into training and test sets
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

Map: 100%|██████████| 3760/3760 [00:01<00:00, 2464.72 examples/s]

{'input_ids': [101, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [38]:
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

In [39]:
train_dataset[0]

{'input_ids': [101,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [40]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=bert_tokenizer, mlm=True, mlm_probability=0.15
)

In [42]:
data_collator([train_dataset[1]])

{'input_ids': tensor([[  101,  1027, 11748,  4801,  4360, 11906,  3523,  1027,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [45]:
training_args = TrainingArguments(
    output_dir="./trained_model",  # Specify the output directory for the trained model
    do_eval=True,
    learning_rate=5e-5,
    num_train_epochs=10,  # Specify the number of training epochs
    per_device_train_batch_size=2,  # Set the batch size for training
    save_total_limit=2,  # Limit the total number of saved checkpoints
    logging_steps = 20

)

# Instantiate the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Start the pre-training
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

In [49]:
!powershell -Command "Invoke-WebRequest -Uri 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BeXRxFT2EyQAmBHvxVaMYQ/bert-scratch-model.pt' -OutFile 'bert-scratch-model.pt'"

In [50]:
model.resize_token_embeddings(30522)
model.load_state_dict(torch.load('bert-scratch-model.pt',map_location=torch.device('cpu')))

<All keys matched successfully>

In [51]:
text = "This is a [MASK] movie!"

In [52]:
mask_filler = pipeline(
    'fill-mask',
    model=model,
    tokenizer=bert_tokenizer
)

In [53]:
result = mask_filler(text)

In [54]:
result

[{'score': 0.049321454018354416,
  'token': 544,
  'token_str': '[unused539]',
  'sequence': 'this is a [unused539] movie!'},
 {'score': 0.04548634961247444,
  'token': 18,
  'token_str': '[unused17]',
  'sequence': 'this is a [unused17] movie!'},
 {'score': 0.034036993980407715,
  'token': 16,
  'token_str': '[unused15]',
  'sequence': 'this is a [unused15] movie!'},
 {'score': 0.024304552003741264,
  'token': 562,
  'token_str': '[unused557]',
  'sequence': 'this is a [unused557] movie!'},
 {'score': 0.02116413414478302,
  'token': 556,
  'token_str': '[unused551]',
  'sequence': 'this is a [unused551] movie!'}]

In [56]:
for result in result:
    print(f"Predicted token: {result['token_str']}, Confidence: {result['score']:.2f}")

Predicted token: [unused539], Confidence: 0.05
Predicted token: [unused17], Confidence: 0.05
Predicted token: [unused15], Confidence: 0.03
Predicted token: [unused557], Confidence: 0.02
Predicted token: [unused551], Confidence: 0.02


In [57]:
pretrained_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
pretrained_tokernizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 488.11it/s, Materializing param=cls.predictions.transform.dense.weight]                 
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [58]:
text = "This is a [MASK] movie!"
mask_filler = pipeline(
    'fill-mask',
    model=pretrained_model,
    tokenizer=pretrained_tokernizer
)

In [59]:
results = mask_filler(text)

In [60]:
results

[{'score': 0.15743982791900635,
  'token': 2307,
  'token_str': 'great',
  'sequence': 'this is a great movie!'},
 {'score': 0.0841229185461998,
  'token': 5469,
  'token_str': 'horror',
  'sequence': 'this is a horror movie!'},
 {'score': 0.08005666732788086,
  'token': 2204,
  'token_str': 'good',
  'sequence': 'this is a good movie!'},
 {'score': 0.048479411751031876,
  'token': 2919,
  'token_str': 'bad',
  'sequence': 'this is a bad movie!'},
 {'score': 0.04168876260519028,
  'token': 10392,
  'token_str': 'fantastic',
  'sequence': 'this is a fantastic movie!'}]